<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Tutorial · DAX en Python: verifica tus medidas
**Analista de datos · Power BI + Tableau** — Recetario con cifras de control y el contexto de filtro en gráficos · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/tutorial_dax.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#dax)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Obtener el valor que debe dar cada medida del recetario con cualquier filtro.
2. Ver por qué el total no es la suma de las filas.
3. Comparar una participación con y sin ALL.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

In [ ]:
# @title Definir las seis medidas (las mismas de DAX)
def medidas(v, m):
    """v: líneas de venta con el filtro de la celda; m: metas con el mismo filtro."""
    ok = v[v.estado != "Anulada"]                      # CALCULATE(..., Ventas[estado] <> "Anulada")
    venta, fact, meta = ok.subtotal.sum(), ok.id_factura.nunique(), m.meta_subtotal.sum()
    return {"Venta neta": venta, "Meta": meta,
            "Cumplimiento": venta / meta if meta else np.nan,          # DIVIDE([Venta neta], [Meta])
            "Facturas": fact,                                            # DISTINCTCOUNT(id_factura)
            "Ticket promedio": venta / fact if fact else np.nan,         # DIVIDE([Venta neta], [Facturas])
            "Margen %": ok.margen_bruto.sum() / venta if venta else np.nan}
ventas_ene = pd.read_csv(ruta("nuevo_mes/ventas_2026_01.csv"), parse_dates=["fecha"])
metas_ene = pd.read_csv(ruta("nuevo_mes/metas_2026_01.csv"))
V = (pd.concat([ventas, ventas_ene]).merge(sedes[["id_sede", "ciudad"]], on="id_sede")
     .merge(clientes[["id_cliente", "segmento"]], on="id_cliente").assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
M = pd.concat([metas, metas_ene]).merge(sedes[["id_sede", "ciudad"]], on="id_sede")
def formato(nombre, x):
    if pd.isna(x): return "(vacío)"
    return porc(x) if nombre in ("Cumplimiento", "Margen %") else miles(x)
display(Markdown("Medidas listas: Venta neta, Meta, Cumplimiento, Facturas, Ticket promedio y Margen %."))

## 1 · El recetario, con tus filtros
Elige los filtros **igual que en las segmentaciones de tu Power BI**. Cada fila muestra la medida en DAX, su equivalente en Python y el valor que debe dar tu tarjeta.

In [ ]:
# @title Recetario con cifras de control
anio = "2025"  # @param ["2024", "2025", "Enero 2026"]
sede = "Todas"  # @param ["Todas", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
canal = "Todos"  # @param ["Todos", "Presencial", "Televentas", "E-commerce"]
a = 2026 if anio == "Enero 2026" else int(anio)
v, m = V[V.anio == a], M[M.anio == a]
if sede != "Todas": v, m = v[v.ciudad == sede], m[m.ciudad == sede]
if canal != "Todos": v = v[v.canal == canal]
ok = v[v.estado != "Anulada"]
base = medidas(v, m)
vaa = V[(V.anio == a - 1) & (V.estado != "Anulada") & (V.mes.isin(v.mes.unique()))]
if sede != "Todas": vaa = vaa[vaa.ciudad == sede]
if canal != "Todos": vaa = vaa[vaa.canal == canal]
rec = [
    ("Venta neta", 'CALCULATE(SUM(Ventas[subtotal]), Ventas[estado] <> "Anulada")', "ok.subtotal.sum()", base["Venta neta"], False),
    ("Venta con anuladas", "SUM(Ventas[subtotal])", "v.subtotal.sum()", v.subtotal.sum(), False),
    ("Líneas", "COUNTROWS(Ventas)", "len(v)", len(v), False),
    ("Facturas", "CALCULATE(DISTINCTCOUNT(Ventas[id_factura]), …)", "ok.id_factura.nunique()", base["Facturas"], False),
    ("Clientes activos", "CALCULATE(DISTINCTCOUNT(Ventas[id_cliente]), …)", "ok.id_cliente.nunique()", ok.id_cliente.nunique(), False),
    ("Ticket promedio", "DIVIDE([Venta neta], [Facturas])", "venta / facturas", base["Ticket promedio"], False),
    ("Margen %", "DIVIDE([Margen bruto], [Venta neta])", "ok.margen_bruto.sum() / venta", base["Margen %"], True),
    ("Costo", "SUMX(FILTER(Ventas, …), Ventas[subtotal] - Ventas[margen_bruto])", "(ok.subtotal - ok.margen_bruto).sum()", (ok.subtotal - ok.margen_bruto).sum(), False),
    ("Meta", "SUM(Metas[meta_subtotal])", "m.meta_subtotal.sum()", base["Meta"], False),
    ("Cumplimiento", "DIVIDE([Venta neta], [Meta])", "venta / meta", base["Cumplimiento"], True),
    ("Venta AA", "CALCULATE([Venta neta], SAMEPERIODLASTYEAR(Calendario[Date]))", "venta del mismo periodo del año anterior", vaa.subtotal.sum() if len(vaa) else np.nan, False),
    ("Var % AA", "DIVIDE([Venta neta] - [Venta AA], [Venta AA])", "venta / venta_aa - 1", base["Venta neta"] / vaa.subtotal.sum() - 1 if len(vaa) else np.nan, True),
]
tabla = pd.DataFrame([{"Medida": r[0], "DAX": r[1], "En Python": r[2], "Debe dar": (porc(r[3], signo=r[0] == "Var % AA") if r[4] else miles(r[3])) if pd.notna(r[3]) else "(vacío)"} for r in rec])
display(tabla.style.hide(axis="index").set_properties(subset=["DAX", "En Python"], **{"font-family": "monospace", "font-size": "11px"}))
if canal != "Todos":
    display(Markdown("**Ojo:** Metas no tiene canal. Con un canal elegido, Meta y Cumplimiento comparan contra la meta completa de la sede."))

## 2 · El total no es la suma de las filas

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Por qué los clientes por canal suman más que el total? |
> | **Qué se grafica** | Barras: clientes activos por canal · Línea: clientes activos en total |
> | **Por qué esta forma** | Pone lado a lado lo que muestra cada fila y lo que muestra el total de una matriz. |
> | **Qué se descartó** | Una tabla: el total parece un error de cálculo. |
> | **Cómo se lee** | Un cliente que compra por dos canales cuenta en dos barras y una sola vez en la línea. El total se calcula con su propio contexto. |
> | **En Excel 365** | Matriz con Ventas[canal] en filas y [Clientes activos] en valores; compara la suma de las filas con la fila Total. |

In [ ]:
# @title Gráfico 1 · Clientes activos por canal y en total
v25 = V[(V.anio == 2025) & (V.estado != "Anulada")]
por_canal = v25.groupby("canal")["id_cliente"].nunique().sort_values()
total = v25.id_cliente.nunique()
fig = go.Figure(go.Bar(x=por_canal.index, y=por_canal.values, marker_color=AZUL, text=por_canal.values, textposition="outside"))
fig.add_hline(y=total, line_dash="dash", line_color=MAGENTA, annotation_text=f"Total: {total}", annotation_position="top left")
fig.update_layout(title=f"Las filas suman {por_canal.sum()} clientes, pero el total es {total}: muchos compran por varios canales",
                  yaxis_title="Clientes activos 2025", height=360, yaxis_range=[0, max(total, por_canal.max()) * 1.2])
fig.show()

## 3 · CALCULATE con y sin ALL

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué cambia si la participación no tiene ALL? |
> | **Qué se grafica** | Barras por sede: participación con ALL(Sedes) y sin ALL |
> | **Por qué esta forma** | Compara las dos versiones de la misma medida en la misma matriz. |
> | **Qué se descartó** | Explicar ALL solo con texto. |
> | **Cómo se lee** | Sin ALL, cada sede se divide por sí misma: 100 % en todas. Con ALL, el denominador es el total y las barras suman 100 %. |
> | **En Excel 365** | Participación = DIVIDE([Venta neta], CALCULATE([Venta neta], ALL(Sedes))). |

In [ ]:
# @title Gráfico 2 · Participación con y sin ALL
version = "Con ALL(Sedes)"  # @param ["Con ALL(Sedes)", "Sin ALL"]
venta_s = v25.groupby("ciudad")["subtotal"].sum().reindex(ORDEN_CIUDADES)
part = venta_s / venta_s.sum() if version == "Con ALL(Sedes)" else venta_s / venta_s
fig = go.Figure(go.Bar(x=part.index, y=part.values, marker_color=AZUL if version.startswith("Con") else MAGENTA,
                       text=[porc(x) for x in part.values], textposition="outside"))
fig.update_layout(title=f"{version}: las barras suman {porc(part.sum())}", yaxis_tickformat=".0%", height=360, yaxis_range=[0, part.max() * 1.2])
fig.show()

## 4 · Retos

In [ ]:
# @title Reto 1 · ¿Cuántos clientes activos tuvo la empresa en 2025?
respuesta = "Elige una opción"  # @param ["Elige una opción", "327", "369", "420", "845"]
revisar(respuesta, str(v25.id_cliente.nunique()), """Es la línea del gráfico 1. 845 es la suma de las filas por canal.""")

In [ ]:
# @title Reto 2 · ¿Qué función DAX quita el filtro de sede del denominador?
respuesta = "Elige una opción"  # @param ["Elige una opción", "SUM", "ALL", "FILTER", "VAR"]
revisar(respuesta, "ALL", """Dentro de CALCULATE, ALL(Sedes) ignora el filtro de sede.""")

In [ ]:
# @title Reto 3 · ¿Qué medida del recetario usa un iterador?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Venta neta", "Costo", "Meta", "Cumplimiento"]
revisar(respuesta, "Costo", """SUMX evalúa subtotal − margen fila por fila y suma los resultados.""")

## Apóyate en IA
Para escribir una medida nueva:

```text
En Power BI tengo Ventas (subtotal, estado, canal, id_cliente, margen_bruto), Metas, Sedes, Clientes y Calendario.
Escribe en DAX la medida [NOMBRE] con VAR y RETURN, sin facturas anuladas, y dime cómo verificarla con una matriz.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Cada medida del recetario tiene su valor de control: compáralo antes de diseñar el informe.
* El total se calcula con su propio contexto; no es la suma de las filas.
* CALCULATE agrega, reemplaza o quita filtros; ALL es la forma de quitarlos.

**Siguiente:** [Tutorial de DAX](https://omarli-360.github.io/powerbi-tableau-analista/#dax) · [Clave 4 · Medidas DAX](https://omarli-360.github.io/powerbi-tableau-analista/#clave-04)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>